# 台股 ML 預測 — 資料前處理

**特徵表示規則**
- 所有特徵：`raw` + `_pct`（橫斷面百分位數）
- 特殊特徵（pe / roe / op_earn / yield_ratio / rev_m_yoy）：額外加 `_bins`（類別，走 Embedding）

**處理順序**
```
資料取得 → EDA 累積分布圖 → 特徵工程
  → build_panel → filter to rev_dates → dropna
  → 計算 label_top / label_bottom（dropna 後，使用訓練宇宙內的 1%）
  → Z-score（只對連續特徵，bins / label 不動）
  → 輸出
```

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install finlab -q
    !apt-get install -y fonts-noto-cjk -qq

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

if 'google.colab' in sys.modules:
    fm._load_fontmanager(try_read_cache=False)
    matplotlib.rcParams['font.family'] = 'Noto Sans CJK JP'
else:
    matplotlib.rcParams['font.family'] = 'DejaVu Sans'

import finlab
from finlab import data
from google.colab import drive, userdata

In [ ]:
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/機器學習/ds_ml_stock/'
os.makedirs(BASE, exist_ok=True)
print(f'BASE: {BASE}')

In [ ]:
finlab.login(userdata.get("finlab"))

## Helper Functions

In [ ]:
def cap_extreme_zscores_robust(df, threshold=3, target_z=3, eps=1e-9):
    """Robust Z-score 橫向異常值清洗（Median + MAD）。"""
    df_new = df.copy()
    med = df_new.median(axis=1)
    mad = (df_new.sub(med, axis=0).abs()).median(axis=1).replace(0, eps)
    robust_z  = df_new.sub(med, axis=0).div(mad * 1.4826, axis=0)
    upper_cap = med + target_z * mad * 1.4826
    lower_cap = med - target_z * mad * 1.4826
    df_new = df_new.mask(robust_z >  threshold, upper_cap, axis=0)
    df_new = df_new.mask(robust_z < -threshold, lower_cap, axis=0)
    return df_new


def cross_sectional_rank_pct(df: pd.DataFrame) -> pd.DataFrame:
    """同一天對所有股票計算橫斷面百分位數排名（0~1），NaN 保留。"""
    return df.rank(axis=1, pct=True, na_option='keep')


def make_bins(df: pd.DataFrame, bins: list, labels: list) -> pd.DataFrame:
    """對 df 全部值套用固定分箱，回傳 float DataFrame（NaN 保留）。"""
    return df.apply(
        lambda col: pd.cut(col, bins=bins, labels=labels, include_lowest=True),
        axis=0
    ).astype(float)


def build_panel_from_feature_dict(feature_label_dict: dict) -> pd.DataFrame:
    """將 {name: DataFrame(date×stock)} 轉為 MultiIndex (datetime, instrument) × features。"""
    series_list = []
    for name, df in feature_label_dict.items():
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"'{name}' is not a DataFrame")
        df = df.copy()
        if df.columns.dtype != 'object':
            df.columns = df.columns.astype(str)
        if not pd.api.types.is_datetime64_any_dtype(df.index):
            df.index = pd.to_datetime(df.index)
        s = df.stack()
        s.name = name
        series_list.append(s)
    panel = pd.concat(series_list, axis=1)
    panel.index = panel.index.set_names(['datetime', 'instrument'])
    return panel.sort_index()


def cross_sectional_zscore(df: pd.DataFrame) -> pd.DataFrame:
    """同一天對所有股票做 Z-score（橫斷面標準化）。
    
    std=0 表示當天所有股票該特徵值完全相同（無變異），
    此時 Z-score 定義為 0（而非 NaN），避免引入缺失值。
    """
    def _z(g):
        mean = g.mean(axis=0)
        std  = g.std(axis=0, ddof=0)
        # std=0 的欄位除出來是 NaN，fillna(0) 補回正確語義
        return g.sub(mean).div(std.where(std > 0, np.nan)).fillna(0)
    return df.groupby(level=0, group_keys=False).apply(_z)

## 資料取得

In [ ]:
with data.universe(market='TSE_OTC'):
    close        = data.get('price:收盤價')
    adj_close    = data.get('etl:adj_close')
    vol          = data.get('price:成交股數')

    roe          = data.get('fundamental_features:ROE稅後')       * (close > 0).reindex(close.index)
    op_earn      = data.get('fundamental_features:營業利益率')     * (close > 0).reindex(close.index)
    rev_m_yoy    = data.get('monthly_revenue:去年同月增減(%)')     * (close > 0).reindex(close.index)
    rev_q_yoy    = data.get('fundamental_features:營收成長率')     * (close > 0).reindex(close.index)
    yield_ratio  = data.get('price_earning_ratio:殖利率(%)')
    market_value = data.get('etl:market_value')
    monthly_rev  = data.get('monthly_revenue:當月營收')            * (close > 0).reindex(close.index)
    eps          = data.get('financial_statement:每股盈餘')
    rev_raw      = data.get('monthly_revenue:當月營收')

## EDA — 原始特徵累積分布圖（前處理前）

用於偵測極端異常值（如月營收年增率 = 999999%）。
橙色虛線 = 1st / 99th percentile，紅色標註 = 實際 min / max。

In [ ]:
eda_features = {
    'roe'          : roe,
    'op_earn'      : op_earn,
    'rev_m_yoy'    : rev_m_yoy,
    'rev_q_yoy'    : rev_q_yoy,
    'yield_ratio'  : yield_ratio,
    'market_value' : market_value,
    'monthly_rev'  : monthly_rev,
    'eps'          : eps,
    'vol'          : vol,
    'close'        : close,
}

n_cols = 2
n_rows = int(np.ceil(len(eda_features) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 3.5))
axes = axes.flatten()

for i, (name, df) in enumerate(eda_features.items()):
    ax = axes[i]

    vals = df.values.flatten()
    vals = vals[np.isfinite(vals)]

    if len(vals) == 0:
        ax.set_title(f'{name}\n(no data)')
        continue

    p01  = np.percentile(vals, 1)
    p99  = np.percentile(vals, 99)
    vmin = vals.min()
    vmax = vals.max()

    # CDF（全範圍，可見極端值）
    sorted_vals = np.sort(vals)
    cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
    ax.plot(sorted_vals, cdf, linewidth=1, color='steelblue')

    # 1st / 99th percentile 參考線
    ax.axvline(p01, color='orange', linestyle='--', linewidth=1, label=f'1%={p01:.1f}')
    ax.axvline(p99, color='orange', linestyle='--', linewidth=1, label=f'99%={p99:.1f}')

    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('value', fontsize=8)
    ax.set_ylabel('CDF', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.2)

    # min / max 標註
    ax.text(0.02, 0.15, f'min={vmin:.2g}', transform=ax.transAxes,
            fontsize=7, color='red')
    ax.text(0.02, 0.05, f'max={vmax:.2g}', transform=ax.transAxes,
            fontsize=7, color='red')
    ax.legend(fontsize=6, loc='lower right')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('原始特徵累積分布圖（前處理前）', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 各特徵關鍵百分位數統計表（快速偵測異常值）
stats_rows = []
for name, df in eda_features.items():
    vals = df.values.flatten()
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        continue
    stats_rows.append({
        'feature' : name,
        'min'     : vals.min(),
        'p1'      : np.percentile(vals, 1),
        'p5'      : np.percentile(vals, 5),
        'median'  : np.median(vals),
        'p95'     : np.percentile(vals, 95),
        'p99'     : np.percentile(vals, 99),
        'max'     : vals.max(),
        'nan_pct' : np.isnan(df.values.flatten()).mean() * 100,
    })

stats_df = pd.DataFrame(stats_rows).set_index('feature')
pd.set_option('display.float_format', '{:.3g}'.format)
display(stats_df)

## 特徵工程

### Step 1 — 計算所有原始衍生特徵

In [ ]:
ret = adj_close.pct_change()

# 乖離
close_5  = close / close.rolling(5).mean()
close_10 = close / close.rolling(10).mean()
close_20 = close / close.rolling(20).mean()
close_60 = close / close.rolling(60).mean()

# 動能偏態
ret_skew_10 = ret.rolling(10).skew()
ret_skew_15 = ret.rolling(15).skew()
ret_skew_21 = ret.rolling(21).skew()
ret_skew_63 = ret.rolling(63).skew()

# 量能
vol_5     = vol / vol.rolling(5).mean()
vol_10    = vol / vol.rolling(10).mean()
vol_20    = vol / vol.rolling(20).mean()
vol_scale = vol * close

# 期間報酬
ret_s_5  = ret.rolling(5).sum()
ret_s_10 = ret.rolling(10).sum()
ret_s_20 = ret.rolling(20).sum()

# 報酬波動
ret_dev_5  = ret.rolling(5).std()
ret_dev_10 = ret.rolling(10).std()
ret_dev_20 = ret.rolling(20).std()

# 估值
eps_4q = eps.rolling(4).sum()
pe     = (close / eps_4q).reindex(close.index)

# 學術動能因子（1M ≈ 21 交易日）
M1, M6, M7, M12, M13, M36 = 21, 126, 147, 252, 273, 756
mom1m  = adj_close / adj_close.shift(M1) - 1
mom12m = adj_close.shift(M1)  / adj_close.shift(M12) - 1
chmom  = (adj_close.shift(M1)  / adj_close.shift(M6)  - 1) - \
         (adj_close.shift(M7)  / adj_close.shift(M12) - 1)
maxret = ret.rolling(M1).max()
mom36m = adj_close.shift(M13) / adj_close.shift(M36) - 1

### Step 2 — 規模特徵：先 cap，再算 rank_pct

In [ ]:
market_value_cap = cap_extreme_zscores_robust(market_value)
vol_scale_cap    = cap_extreme_zscores_robust(vol_scale)
monthly_rev_cap  = cap_extreme_zscores_robust(monthly_rev)

market_value_pct = cross_sectional_rank_pct(market_value_cap)
vol_scale_pct    = cross_sectional_rank_pct(vol_scale_cap)
monthly_rev_pct  = cross_sectional_rank_pct(monthly_rev_cap)

### Step 3 — 業績特徵：cap 後再算 rank_pct

In [ ]:
rev_m_yoy_clean = cap_extreme_zscores_robust(rev_m_yoy)
rev_q_yoy_clean = cap_extreme_zscores_robust(rev_q_yoy)

rev_m_yoy_pct   = cross_sectional_rank_pct(rev_m_yoy_clean)
rev_q_yoy_pct   = cross_sectional_rank_pct(rev_q_yoy_clean)

### Step 4 — 其餘特徵的 rank_pct

In [ ]:
close_5_pct,  close_10_pct  = cross_sectional_rank_pct(close_5),  cross_sectional_rank_pct(close_10)
close_20_pct, close_60_pct  = cross_sectional_rank_pct(close_20), cross_sectional_rank_pct(close_60)

ret_skew_10_pct = cross_sectional_rank_pct(ret_skew_10)
ret_skew_15_pct = cross_sectional_rank_pct(ret_skew_15)
ret_skew_21_pct = cross_sectional_rank_pct(ret_skew_21)
ret_skew_63_pct = cross_sectional_rank_pct(ret_skew_63)

vol_5_pct,  vol_10_pct, vol_20_pct = (
    cross_sectional_rank_pct(vol_5),
    cross_sectional_rank_pct(vol_10),
    cross_sectional_rank_pct(vol_20),
)

ret_pct     = cross_sectional_rank_pct(ret)
ret_s_5_pct = cross_sectional_rank_pct(ret_s_5)
ret_s_10_pct= cross_sectional_rank_pct(ret_s_10)
ret_s_20_pct= cross_sectional_rank_pct(ret_s_20)

ret_dev_5_pct  = cross_sectional_rank_pct(ret_dev_5)
ret_dev_10_pct = cross_sectional_rank_pct(ret_dev_10)
ret_dev_20_pct = cross_sectional_rank_pct(ret_dev_20)

pe_pct          = cross_sectional_rank_pct(pe)
yield_ratio_pct = cross_sectional_rank_pct(yield_ratio)
roe_pct         = cross_sectional_rank_pct(roe)
op_earn_pct     = cross_sectional_rank_pct(op_earn)

mom1m_pct  = cross_sectional_rank_pct(mom1m)
mom12m_pct = cross_sectional_rank_pct(mom12m)
chmom_pct  = cross_sectional_rank_pct(chmom)
maxret_pct = cross_sectional_rank_pct(maxret)
mom36m_pct = cross_sectional_rank_pct(mom36m)

### Step 5 — 特殊特徵的 bins（類別，後續走 Embedding）

In [ ]:
# PE：0=虧損 1=超便宜 2=合理 3=稍貴 4=成長溢價 5=極貴
pe_bins = make_bins(pe, [-np.inf,0,10,20,30,50,np.inf], [0,1,2,3,4,5])

# ROE(%)：0=虧損 1=微利 2=低獲利 3=健康 4=優秀 5=卓越
roe_bins = make_bins(roe, [-np.inf,0,5,12,20,30,np.inf], [0,1,2,3,4,5])

# 營業利益率(%)：0=虧損 1=薄利 2=普通 3=良好 4=優秀 5=極高
op_earn_bins = make_bins(op_earn, [-np.inf,0,5,15,25,40,np.inf], [0,1,2,3,4,5])

# 殖利率(%)：0=無股利 1=低 2=普通 3=良好 4=高 5=極高
yield_ratio_bins = make_bins(yield_ratio, [-np.inf,0,2,4,6,8,np.inf], [0,1,2,3,4,5])

# 月營收年增率(%)：0=大幅衰退 1=溫和衰退 2=持平 3=溫和成長 4=強勁 5=爆發
rev_m_yoy_bins = make_bins(rev_m_yoy_clean, [-np.inf,-20,-5,5,20,50,np.inf], [0,1,2,3,4,5])

## Label 建構

In [ ]:
rev_dates_raw = rev_raw.index_str_to_date().index
trade_dates   = close.index
rev_dates     = rev_dates_raw.intersection(trade_dates).sort_values()

print(f"月營收公布日（交集後）共 {len(rev_dates)} 個")
print(f"範圍：{rev_dates[0]} ~ {rev_dates[-1]}")

In [ ]:
# 原始 forward return（用於後續計算 binary label，不直接作為訓練 label）
adj_close_rev = adj_close.reindex(rev_dates)
label_rev     = adj_close_rev.shift(-1) / adj_close_rev - 1

# reindex 回全交易日 index，讓 label 能與其他特徵在 build_panel 時對齊
# 非 rev_dates 的交易日 label 為 NaN，後續 filter + dropna 會清掉
label         = label_rev.reindex(close.index)

## Panel 建構

In [ ]:
feature_label_dict = {
    # 乖離
    'close_5'       : close_5,         'close_5_pct'       : close_5_pct,
    'close_10'      : close_10,        'close_10_pct'      : close_10_pct,
    'close_20'      : close_20,        'close_20_pct'      : close_20_pct,
    'close_60'      : close_60,        'close_60_pct'      : close_60_pct,
    # 動能偏態
    'ret_skew_10'   : ret_skew_10,     'ret_skew_10_pct'   : ret_skew_10_pct,
    'ret_skew_15'   : ret_skew_15,     'ret_skew_15_pct'   : ret_skew_15_pct,
    'ret_skew_21'   : ret_skew_21,     'ret_skew_21_pct'   : ret_skew_21_pct,
    'ret_skew_63'   : ret_skew_63,     'ret_skew_63_pct'   : ret_skew_63_pct,
    # 業績動能
    'rev_m_yoy'     : rev_m_yoy_clean, 'rev_m_yoy_pct'     : rev_m_yoy_pct,
    'rev_m_yoy_bins': rev_m_yoy_bins,
    'rev_q_yoy'     : rev_q_yoy_clean, 'rev_q_yoy_pct'     : rev_q_yoy_pct,
    # 量能
    'vol_5'         : vol_5,           'vol_5_pct'         : vol_5_pct,
    'vol_10'        : vol_10,          'vol_10_pct'        : vol_10_pct,
    'vol_20'        : vol_20,          'vol_20_pct'        : vol_20_pct,
    # 期間報酬
    'ret'           : ret,             'ret_pct'           : ret_pct,
    'ret_s_5'       : ret_s_5,         'ret_s_5_pct'       : ret_s_5_pct,
    'ret_s_10'      : ret_s_10,        'ret_s_10_pct'      : ret_s_10_pct,
    'ret_s_20'      : ret_s_20,        'ret_s_20_pct'      : ret_s_20_pct,
    # 報酬波動
    'ret_dev_5'     : ret_dev_5,       'ret_dev_5_pct'     : ret_dev_5_pct,
    'ret_dev_10'    : ret_dev_10,      'ret_dev_10_pct'    : ret_dev_10_pct,
    'ret_dev_20'    : ret_dev_20,      'ret_dev_20_pct'    : ret_dev_20_pct,
    # 估值
    'pe'            : pe,              'pe_pct'            : pe_pct,
    'pe_bins'       : pe_bins,
    'yield_ratio'   : yield_ratio,     'yield_ratio_pct'   : yield_ratio_pct,
    'yield_ratio_bins': yield_ratio_bins,
    # 基本面
    'roe'           : roe,             'roe_pct'           : roe_pct,
    'roe_bins'      : roe_bins,
    'op_earn'       : op_earn,         'op_earn_pct'       : op_earn_pct,
    'op_earn_bins'  : op_earn_bins,
    # 規模
    'market_value'  : market_value_cap, 'market_value_pct' : market_value_pct,
    'vol_scale'     : vol_scale_cap,    'vol_scale_pct'    : vol_scale_pct,
    'monthly_rev'   : monthly_rev_cap,  'monthly_rev_pct'  : monthly_rev_pct,
    # 學術動能
    'mom1m'         : mom1m,           'mom1m_pct'         : mom1m_pct,
    'mom12m'        : mom12m,          'mom12m_pct'        : mom12m_pct,
    'chmom'         : chmom,           'chmom_pct'         : chmom_pct,
    'maxret'        : maxret,          'maxret_pct'        : maxret_pct,
    'mom36m'        : mom36m,          'mom36m_pct'        : mom36m_pct,
    # Label（原始 forward return，後續從此計算 binary label）
    'label'         : label,
}

In [ ]:
panel = build_panel_from_feature_dict(feature_label_dict)

# 將欄位分為三類，後續各自處理：
#   bins_cols  → 類別特徵（走 Embedding，需要整數 index）
#   label_cols → forward return，用來計算 binary label，不做 Z-score
#   cont_cols  → 連續特徵，後續做橫斷面 Z-score
bins_cols  = [c for c in panel.columns if c.endswith('_bins')]
label_cols = ['label']
cont_cols  = [c for c in panel.columns if c not in bins_cols and c not in label_cols]

print(f"Panel shape（全交易日）: {panel.shape}")
print(f"連續特徵: {len(cont_cols)}  類別特徵(bins): {len(bins_cols)}")

## 前處理 Pipeline

In [ ]:
# Step 1: 過濾至 rev_dates
mask        = panel.index.get_level_values('datetime').isin(rev_dates)
panel_rev   = panel[mask]
print(f"[Step 1] Filter to rev_dates → {panel_rev.shape}")

# Step 2: 移除含 NaN 的 row
panel_clean = panel_rev.dropna(how='any')
print(f"[Step 2] dropna            → {panel_clean.shape}")
print(f"         日期數: {panel_clean.index.get_level_values('datetime').nunique()}")
print(f"         平均每日股票數: {panel_clean.groupby(level='datetime').size().mean():.0f}")

In [ ]:
# Step 3: 計算 binary label（必須在 dropna 後才算）
# 原因：dropna 後才是完整的訓練宇宙，確保「前/後 1%」的邊界是在有效樣本內算的
# 若在 dropna 前算，部分 NaN 股票被排除後 1% 邊界會偏移
panel_clean = panel_clean.copy()

panel_clean['label_top'] = (
    panel_clean.groupby(level='datetime')['label']
    .transform(lambda x: (x.rank(pct=True) >= 0.99).astype(float))
)
panel_clean['label_bottom'] = (
    panel_clean.groupby(level='datetime')['label']
    .transform(lambda x: (x.rank(pct=True) <= 0.01).astype(float))
)

label_cols_all = ['label', 'label_top', 'label_bottom']

# 正類比例應接近 1%（微幅偏差來自每天股票數不能整除）
top_ratio = panel_clean['label_top'].mean()
bot_ratio = panel_clean['label_bottom'].mean()
print(f"label_top    正類比例: {top_ratio:.3%}")
print(f"label_bottom 正類比例: {bot_ratio:.3%}")

In [ ]:
# Step 4: 三類欄位分別處理後合併
#   cont_cols  → 橫斷面 Z-score（去除量綱差異，統一尺度）
#   bins_cols  → 轉 int（Embedding 需要整數 index，不做標準化）
#   label_cols → 原樣保留（binary 0/1 或 forward return，做 Z-score 無意義）
panel_z      = cross_sectional_zscore(panel_clean[cont_cols])
panel_bins   = panel_clean[bins_cols].astype(int)
panel_labels = panel_clean[label_cols_all]

panel_final = pd.concat([panel_z, panel_bins, panel_labels], axis=1)
print(f"[Step 4] Z-score + concat  → {panel_final.shape}")

## 資料除錯 — 零變異特徵檢查

Z-score 後若某特徵仍含 NaN，代表該特徵在某天所有股票值完全相同（std=0）。
此區塊顯示有哪些特徵出現此狀況、各出現幾天。

## 輸出

In [ ]:
X        = panel_final.drop(columns=label_cols_all)
y_top    = panel_final['label_top'].astype(int)
y_bottom = panel_final['label_bottom'].astype(int)
y_return = panel_final['label']

print(f"X shape       : {X.shape}")
print(f"y_top shape   : {y_top.shape}  （正類: {y_top.sum():,} 筆）")
print(f"y_bottom shape: {y_bottom.shape}  （正類: {y_bottom.sum():,} 筆）")
print(f"\n連續特徵 ({len(cont_cols)}):")
for c in cont_cols: print(f"  {c}")
print(f"\n類別特徵 ({len(bins_cols)}):")
for c in bins_cols: print(f"  {c}")

In [ ]:
X.to_parquet(BASE + 'X_features.parquet')
y_top.to_frame().to_parquet(BASE + 'y_top.parquet')
y_bottom.to_frame().to_parquet(BASE + 'y_bottom.parquet')
y_return.to_frame().to_parquet(BASE + 'y_return.parquet')

print("✅ 儲存完成")
print(f"   {BASE}X_features.parquet  → {X.shape}")
print(f"   {BASE}y_top.parquet       → top 1% binary label")
print(f"   {BASE}y_bottom.parquet    → bottom 1% binary label")
print(f"   {BASE}y_return.parquet    → raw forward return（分析用）")
print(f"   日期範圍: {panel_final.index.get_level_values('datetime').min()} ~ "
      f"{panel_final.index.get_level_values('datetime').max()}")